# 01 — Data overview

Loads the Parquet files under `data/raw/` (written by `python -m prisma.ingest`) and saves three figures to `results/figures/`:

1. `annual_tmax_by_location.png` — annual mean daily maximum temperature per location, ERA5 1996–2025 (5-year rolling mean in bold).
2. `ensemble_forecast_band.png` — current multi-model ensemble forecast for Córdoba with the 10–90% band.
3. `tmax_anomaly_heatmap_cordoba.png` — monthly temperature-maximum anomaly for Córdoba against its own calendar-month climatology.

## What this does not show

- **No forecast verification.** Nothing here compares a forecast against what actually happened; CRPS, Brier and reliability do not exist yet (Phase 1).
- **No calibration.** The band is the raw ensemble spread, not a calibrated interval.
- **ERA5 is gridded reanalysis, not a station measurement.** Station observations come from GHCN-Daily (ingested, not plotted here).
- **One issue time only.** A single ensemble snapshot says nothing about reliability; that requires the daily archive.
- **Anomalies are relative to 1996–2025**, not to a pre-industrial or policy reference period.

In [ ]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "sources.yaml").exists():
            return candidate
    raise FileNotFoundError("run this notebook from inside the repository")


ROOT = find_repo_root()
DATA = ROOT / "data" / "raw"
FIGS = ROOT / "results" / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

era5 = pd.read_parquet(DATA / "openmeteo-era5.parquet")
ens = pd.read_parquet(DATA / "openmeteo-ensemble.parquet")
era5["time"] = pd.to_datetime(era5["time"])
ens["time"] = pd.to_datetime(ens["time"])

era5.shape, ens.shape

In [ ]:
tmax = era5.loc[era5["variable"] == "t2m_max", ["location", "time", "value"]].copy()
tmax["year"] = tmax["time"].dt.year
annual = (
    tmax.loc[tmax["year"] <= 2025]
    .groupby(["location", "year"], as_index=False)["value"]
    .mean()
    .sort_values("year")
)

fig, ax = plt.subplots(figsize=(11, 6))
cmap = plt.get_cmap("tab20")
locations = sorted(annual["location"].unique())
for i, loc_name in enumerate(locations):
    group = annual.loc[annual["location"] == loc_name]
    color = cmap(i % 20)
    ax.plot(group["year"], group["value"], color=color, lw=0.7, alpha=0.35)
    roll = group.set_index("year")["value"].rolling(5, min_periods=3).mean()
    ax.plot(roll.index, roll.values, color=color, lw=1.8, alpha=0.95, label=loc_name)

ax.set_xlabel("Year")
ax.set_ylabel("Annual mean daily maximum temperature (°C)")
ax.set_title(
    "Annual mean daily maximum temperature by location\n"
    "ERA5 1996–2025; thin line = annual mean, bold = 5-year rolling mean"
)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=7, frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGS / "annual_tmax_by_location.png", dpi=150)
plt.close(fig)

In [ ]:
loc_name = "Cordoba Aero"
subset = ens.loc[
    (ens["location"] == loc_name) & ens["variable"].isin(["t2m_max", "t2m_min"])
]
if subset.empty:
    raise FileNotFoundError(f"no ensemble rows for {loc_name}; run python -m prisma.ingest")

fig, ax = plt.subplots(figsize=(10, 5.5))
styles = {
    "t2m_max": ("#c44e52", "Daily maximum"),
    "t2m_min": ("#4c72b0", "Daily minimum"),
}
for var, (color, label) in styles.items():
    pivot = subset.loc[subset["variable"] == var].pivot_table(
        index="time", columns=["model", "member"], values="value"
    )
    for column in pivot.columns:
        ax.plot(pivot.index, pivot[column], color=color, lw=0.4, alpha=0.15)
    p10 = pivot.quantile(0.1, axis=1)
    p50 = pivot.median(axis=1)
    p90 = pivot.quantile(0.9, axis=1)
    ax.fill_between(pivot.index, p10, p90, color=color, alpha=0.2, label=f"{label} 10–90% band")
    ax.plot(pivot.index, p50, color=color, lw=2, label=f"{label} median")

issue = pd.to_datetime(subset["issue_time"]).max()
ax.set_title(
    f"Open-Meteo multi-model ensemble forecast — {loc_name}\n"
    f"issue time {issue:%Y-%m-%d %H:%M} UTC (fetch-time proxy)"
)
ax.set_xlabel("Forecast date (local calendar day)")
ax.set_ylabel("Temperature (°C)")
ax.legend(fontsize=8, frameon=False, ncol=2)
ax.grid(alpha=0.25)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGS / "ensemble_forecast_band.png", dpi=150)
plt.close(fig)

In [ ]:
series = (
    era5.loc[(era5["location"] == loc_name) & (era5["variable"] == "t2m_max")]
    .set_index("time")["value"]
    .sort_index()
)
if series.empty:
    raise FileNotFoundError(f"no ERA5 rows for {loc_name}; run python -m prisma.ingest")

monthly = series.resample("ME").mean()
climatology = monthly.groupby(monthly.index.month).mean()
anomaly = monthly - monthly.index.month.map(climatology)
grid = (
    pd.DataFrame(
        {"year": anomaly.index.year, "month": anomaly.index.month, "anom": anomaly.to_numpy()}
    )
    .pivot(index="year", columns="month", values="anom")
    .sort_index()
)

fig, ax = plt.subplots(figsize=(10, 7))
vmax = float(grid.abs().quantile(0.98).max())
im = ax.imshow(
    grid.to_numpy(),
    aspect="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    extent=[0.5, 12.5, grid.index.max() + 0.5, grid.index.min() - 0.5],
)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)
ax.set_ylabel("Year")
ax.set_title(
    f"Monthly mean daily maximum temperature anomaly — {loc_name}\n"
    "vs 1996–2025 calendar-month climatology (ERA5)"
)
cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Anomaly (°C)")
fig.tight_layout()
fig.savefig(FIGS / "tmax_anomaly_heatmap_cordoba.png", dpi=150)
plt.close(fig)

## Regenerating

From the repository root:

```bash
uv sync --project backend
uv run --project backend python -m prisma.ingest
uv run --project backend python scripts/run_notebook.py
```

The figures land in `results/figures/`. The Parquet files under `data/raw/` are downloaded and gitignored.